In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# !pip install -q huggingface_hub
# from huggingface_hub import login

# login()

In [16]:
!pip install -q datasets pandas pyarrow tqdm

In [17]:
import os
import re
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset

# =========================
# Настройки
# =========================

BASE_DIR = "/content/drive/MyDrive/MLops_project/project"

RAW_DIR = f"{BASE_DIR}/data/raw/wb_feedbacks_full"
CLEAN_DIR = f"{BASE_DIR}/data/interim/wb_feedbacks_clean_full"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)

DATASET_NAME = "nyuuzyou/wb-feedbacks"
SPLIT = "train"

CHUNK_SIZE = 100_000
MAX_CHARS = 5000

# Для полного скачивания:
MAX_ROWS = None

# Для теста можно поставить, например:
# MAX_ROWS = 500_000

def clean_text(text, max_chars=5000):
    """
    Минимальная предобработка:
    - NaN -> пустая строка
    - переносы строк -> пробел
    - табы -> пробел
    - странные пробельные символы -> обычный пробел
    - несколько пробелов -> один пробел
    - strip
    - обрезка до max_chars символов
    """
    if pd.isna(text):
        return ""

    text = str(text)

    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = text.replace("\t", " ")

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    text = text[:max_chars]

    return text

In [18]:
# =========================
# Загрузка датасета в streaming-режиме
# =========================

ds = load_dataset(
    DATASET_NAME,
    split=SPLIT,
    streaming=True
)

buffer = []
chunk_id = 0

total_raw_rows = 0
total_clean_rows = 0

# Если MAX_ROWS задан, progress bar будет с известным total.
# Если MAX_ROWS = None, progress bar будет просто показывать количество обработанных строк.
pbar = tqdm(
    total=MAX_ROWS,
    desc="Downloading and processing rows",
    unit="rows"
)

for row in ds:
    buffer.append(row)
    total_raw_rows += 1
    pbar.update(1)

    if len(buffer) >= CHUNK_SIZE:
        df_raw = pd.DataFrame(buffer)

        # =========================
        # 1. Сохраняем сырой чанк
        # =========================
        raw_path = f"{RAW_DIR}/raw_part_{chunk_id:05d}.parquet"
        df_raw.to_parquet(raw_path, index=False)

        # =========================
        # 2. Минимальная предобработка
        # =========================
        df_clean = df_raw.copy()

        # Если колонки text почему-то нет, создаем пустую
        if "text" not in df_clean.columns:
            df_clean["text"] = ""

        # Чистим только колонку text
        df_clean["text"] = df_clean["text"].apply(
            lambda x: clean_text(x, max_chars=MAX_CHARS)
        )

        # Убираем полностью пустые отзывы
        df_clean = df_clean[df_clean["text"] != ""].copy()

        clean_path = f"{CLEAN_DIR}/clean_part_{chunk_id:05d}.parquet"
        df_clean.to_parquet(clean_path, index=False)

        total_clean_rows += len(df_clean)

        tqdm.write(
            f"Saved chunk {chunk_id:05d} | "
            f"raw_rows={len(df_raw)} | "
            f"clean_rows={len(df_clean)}"
        )

        buffer = []
        chunk_id += 1

    if MAX_ROWS is not None and total_raw_rows >= MAX_ROWS:
        break


# =========================
# Сохраняем последний неполный чанк
# =========================

if len(buffer) > 0:
    df_raw = pd.DataFrame(buffer)

    raw_path = f"{RAW_DIR}/raw_part_{chunk_id:05d}.parquet"
    df_raw.to_parquet(raw_path, index=False)

    df_clean = df_raw.copy()

    if "text" not in df_clean.columns:
        df_clean["text"] = ""

    df_clean["text"] = df_clean["text"].apply(
        lambda x: clean_text(x, max_chars=MAX_CHARS)
    )

    df_clean = df_clean[df_clean["text"] != ""].copy()

    clean_path = f"{CLEAN_DIR}/clean_part_{chunk_id:05d}.parquet"
    df_clean.to_parquet(clean_path, index=False)

    total_clean_rows += len(df_clean)

    tqdm.write(
        f"Saved chunk {chunk_id:05d} | "
        f"raw_rows={len(df_raw)} | "
        f"clean_rows={len(df_clean)}"
    )

pbar.close()

print("Готово")
print(f"Всего сырых строк обработано: {total_raw_rows}")
print(f"Всего строк после минимальной предобработки: {total_clean_rows}")
print(f"Сырые данные сохранены в: {RAW_DIR}")
print(f"Очищенные данные сохранены в: {CLEAN_DIR}")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Saved chunk 00000 | raw_rows=100000 | clean_rows=100000
Saved chunk 00001 | raw_rows=100000 | clean_rows=100000
Saved chunk 00002 | raw_rows=100000 | clean_rows=100000
Saved chunk 00003 | raw_rows=100000 | clean_rows=100000
Saved chunk 00004 | raw_rows=100000 | clean_rows=100000
Saved chunk 00005 | raw_rows=100000 | clean_rows=100000
Saved chunk 00006 | raw_rows=100000 | clean_rows=100000
Saved chunk 00007 | raw_rows=100000 | clean_rows=100000
Saved chunk 00008 | raw_rows=100000 | clean_rows=100000
Saved chunk 00009 | raw_rows=100000 | clean_rows=100000
Saved chunk 00010 | raw_rows=100000 | clean_rows=100000
Saved chunk 00011 | raw_rows=100000 | clean_rows=100000
Saved chunk 00012 | raw_rows=100000 | clean_rows=100000
Saved chunk 00013 | raw_rows=100000 | clean_rows=100000
Saved chunk 00014 | raw_rows=100000 | clean_rows=100000
Saved chunk 00015 | raw_rows=100000 | clean_rows=100000
Saved chunk 00016 | raw_rows=100000 | clean_rows=100000
Saved chunk 00017 | raw_rows=100000 | clean_rows

In [19]:
sample_path = f"{CLEAN_DIR}/clean_part_00000.parquet"

df_sample = pd.read_parquet(sample_path)

print(df_sample.shape)
df_sample.head()

(100000, 5)


,nmId,productValuation,color,text,answer
0,0,1,,"5 попыток было испечь хлеб с этой муки, и один...",
1,0,1,,1 звезда за то что нет даты изготовления и пит...,Срок годности может стоять вне окошка. Проверь...
2,0,1,,"2 недели уже стоит, так ничего и не взошло",Добрый день! Некоторым растениям иногда требуе...
3,0,1,,25.07.21 получил данный товар. Упаковка была н...,
4,0,1,,2x увелечение.,
